# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
%load_ext dotenv
%dotenv 



In [2]:
import dask.dataframe as dd

c:\Users\kaurp\miniconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 11.0.0. Please consider upgrading.
  warnings.warn(


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Get the directory path from the environment variable
price_data = os.getenv('PRICE_DATA')

# Use glob to find all parquet files in the directory
parquet_files = glob(os.path.join(price_data, "**/*.parquet"), recursive=True)

parquet_files


['../../05_src/data/prices\\AAPL\\AAPL_2000\\part.0.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2000\\part.1.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2001\\part.0.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2001\\part.1.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2002\\part.0.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2002\\part.1.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2003\\part.0.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2003\\part.1.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2004\\part.0.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2004\\part.1.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2005\\part.0.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2005\\part.1.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2006\\part.0.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2006\\part.1.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2007\\part.0.parquet',
 '../../05_src/data/prices\\AAPL\\AAPL_2007\\part.1.parquet',
 '../../

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [4]:
# Read the parquet files using Dask
dd_px = dd.read_parquet(parquet_files)

dd_px = dd_px.reset_index()

# Using apply() to create 'Close_lag_1' and 'Adj_Close_lag_1'
dd_shift = dd_px.groupby('Ticker', group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1=x['Close'].shift(1),
        Adj_Close_lag_1=x['Adj Close'].shift(1)  # Corrected column name
    )
)

# Calculate returns based on 'Close'
dd_returns = dd_shift.assign(
    Returns = lambda x: x['Close'] / x['Close_lag_1'] - 1
)

# Calculate high-low range
dd_returns['hi_lo_range'] = dd_returns['High'] - dd_returns['Low']

# Selecting relevant columns
dd_feat = dd_returns[['Ticker', 'Date', 'Close', 'Adj Close', 'Returns', 'hi_lo_range']]

dd_feat



C:\Users\kaurp\AppData\Local\Temp\ipykernel_13404\2102195971.py:7: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_shift = dd_px.groupby('Ticker', group_keys=False).apply(


,Ticker,Date,Close,Adj Close,Returns,hi_lo_range
npartitions=3328,,,,,,
,object,"datetime64[ns, UTC]",float64,float64,float64,float64
,...,...,...,...,...,...
...,...,...,...,...,...,...
,...,...,...,...,...,...
,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [6]:
# Convert the Dask DataFrame to a Pandas DataFrame
dd_feat_pd = dd_feat.compute()

dd_feat_pd = dd_feat_pd.reset_index(drop=True)

# Step 2: Calculate 10-day rolling mean of returns for each ticker
dd_rolling = dd_feat.groupby('Ticker', group_keys=False).apply(
lambda x: x.assign(rolling_10 = x['Returns'].rolling(10).mean())
)

dd_rolling



C:\Users\kaurp\AppData\Local\Temp\ipykernel_13404\1006709583.py:7: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_rolling = dd_feat.groupby('Ticker', group_keys=False).apply(


,Ticker,Date,Close,Adj Close,Returns,hi_lo_range,rolling_10
npartitions=3328,,,,,,,
,object,"datetime64[ns, UTC]",float64,float64,float64,float64,float64
,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...
,...,...,...,...,...,...,...


Please comment:

Was it necessary to convert to pandas to calculate the moving average return? No, it was not strictly necessary to convert to Pandas to calculate the moving average return, but it is often more straightforward when working with rolling windows in Dask. Dask currently has limited support for rolling windows, and when applied to Dask DataFrames, the rolling operations can be complex and may not be as efficient as those in Pandas.

Would it have been better to do it in Dask? Why? Yes, it would be better to perform the operation in Dask if we are working with a large dataset that cannot fit into memory. Dask is designed for parallel computing and can efficiently handle larger-than-memory datasets by processing them in chunks across multiple cores or machines. Using Dask would allow us to scale the computation, avoiding memory issues and improving performance for large datasets.

However, if the dataset is small enough to fit into memory, converting to Pandas can be simpler and more efficient for operations like rolling windows, as Pandas has direct support for such operations

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.